# 05 — Agent Simulation (clean run)

Shin-Carley Framework: TPM fatigue + JP regression + CueStrength FPL + F_dynamic accumulation

Run all cells top to bottom. Ollama must be running (`ollama serve`).

In [ ]:
import sys, os
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

load_dotenv(ROOT / '.env')

from src.agent import Agent
from src.ollama_extractor import OllamaExtractor
from src.decision_loop import simulate_email
from src.simulation import run_simulation, save_results, click_rate_summary

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', None)

print('All imports OK')
print(f'Root: {ROOT}')

In [ ]:
# Ollama check — using llama3.1:8b
_oe = OllamaExtractor(model='llama3.1:8b')
OLLAMA_OK = _oe.is_available()
if OLLAMA_OK:
    print('Ollama OK — llama3.1:8b ready.')
else:
    print('Ollama not reachable. Will use cached cues (still works).')

In [ ]:
# Single agent spot-check — verify formulas are working
agent = Agent.random_agent('demo_agent', seed=7)

print('=== Agent traits ===')
print(f'  age={agent.age:.1f}, education={agent.education_level:.1f}')
print(f'  job_complexity={agent.job_complexity:.1f}, intrinsic_motivation={agent.intrinsic_motivation:.1f}')
print(f'  suspicion_threshold={agent.suspicion_threshold}, max_cues_processed={agent.max_cues_processed}')
print()

print(f'{"Hour":>6}  {"KSS":>6}  {"f_dyn":>6}  {"TotFat":>7}  {"FinalJP":>8}  {"FPL":>6}')
print('-' * 55)
for hour in [8.0, 10.0, 12.0, 14.0, 16.0]:
    agent.advance_workday(hour)
    print(f'{int(hour):>4}:00  {agent.compute_kss():>6.2f}  {agent.f_dynamic:>6.3f}  '
          f'{agent.compute_total_fatigue():>7.3f}  {agent.compute_job_performance():>8.3f}  '
          f'{agent.compute_flawed_perception_level():>6.3f}')

In [ ]:
# CueStrength demo — show how FPL varies per cue type
agent.advance_workday(8.0)
base = agent.compute_flawed_perception_level()
print(f'Base FPL at 8am: {base:.3f}')
print()
print(f'{"Cue":<22}  {"Strength":>8}  {"Cue FPL":>8}  {"% of base":>9}')
print('-' * 52)
from src.agent import _CUE_STRENGTH
for cue, strength in sorted(_CUE_STRENGTH.items(), key=lambda x: -x[1]):
    cfpl = agent.get_cue_fpl(cue)
    print(f'{cue:<22}  {strength:>8.1f}  {cfpl:>8.3f}  {cfpl/base*100:>8.1f}%')

In [ ]:
results_path = ROOT / 'data' / 'simulation_results.csv'

# Set RERUN=True to re-run simulation, False to load existing results
RERUN      = True
USE_OLLAMA = True   # uses llama3.1:8b with cached results

if USE_OLLAMA:
    import src.simulation as _sim_mod
    _sim_mod.CueExtractor = OllamaExtractor
    print('Cue extraction: OllamaExtractor (llama3.1:8b) — cache-first')
else:
    print('Cue extraction: cached only')

if not results_path.exists() or RERUN:
    df = run_simulation(
        emails_csv=str(ROOT / 'data' / 'processed' / 'master_emails.csv'),
        n_agents=30,
        workday_hours=[8.0, 10.0, 12.0, 14.0, 16.0],
        seed=42,
        cache_dir=str(ROOT / 'data' / 'cue_cache'),
    )
    save_results(df, str(results_path))
else:
    df = pd.read_csv(results_path)
    print(f'Loaded existing results: {len(df):,} rows')

df.head()

In [ ]:
# Cue extraction quality check
cue_stats = (
    df[['email_id', 'source', 'cues_extracted']]
    .drop_duplicates('email_id')
    .groupby('source')['cues_extracted']
    .agg(['mean', 'min', 'max', 'count'])
    .round(2)
)
print('Avg cues per email by source (Ollama):')
print(cue_stats)

In [ ]:
# Click rates by source and hour
phishing_df = df[df['actual_class'] == 1].copy()
phishing_df['clicked'] = (phishing_df['decision'] == 'clicked').astype(int)

by_source_hour = (
    phishing_df.groupby(['source', 'workday_hour'])['clicked']
    .mean().reset_index()
    .rename(columns={'clicked': 'click_rate'})
)

fig, ax = plt.subplots(figsize=(9, 5))
for source, grp in by_source_hour.groupby('source'):
    ax.plot(grp['workday_hour'], grp['click_rate'], marker='o', label=source)
ax.set_xlabel('Workday hour')
ax.set_ylabel('Click rate (phishing)')
ax.set_title('Phishing click rate across the workday — by email source')
ax.legend()
ax.set_xticks([8, 10, 12, 14, 16])
ax.set_xticklabels(['8am', '10am', '12pm', '2pm', '4pm'])
plt.tight_layout()
plt.savefig(ROOT / 'results' / 'click_rate_by_source.png', dpi=150)
plt.show()

In [ ]:
# Summary table
summary = click_rate_summary(df)
print('Phishing click rate by source (averaged across hours):')
avg_by_source = (
    phishing_df.groupby('source')['clicked'].mean().sort_values(ascending=False)
)
print(avg_by_source.to_string())

print()
print('Benign pass rate (agents correctly letting through legitimate email):')
benign_df = df[df['actual_class'] == 0].copy()
benign_df['passed'] = (benign_df['decision'] == 'clicked').astype(int)
print(f"  {benign_df['passed'].mean():.1%}")

In [ ]:
# Fatigue effect — controlled experiment (threshold=2 agents only)
low_thresh = phishing_df[phishing_df['suspicion_threshold'] == 2].copy()
low_thresh['fatigue_bin'] = pd.qcut(
    low_thresh['total_fatigue'], q=3, labels=['Low', 'Medium', 'High']
)

fatigue_click = (
    low_thresh.groupby('fatigue_bin', observed=True)['clicked'].mean()
)
print('Click rate by fatigue level (threshold=2 agents, controlled experiment):')
print(fatigue_click.to_string())

fig, ax = plt.subplots(figsize=(6, 4))
fatigue_click.plot(kind='bar', ax=ax, color=['#2ecc71', '#f39c12', '#e74c3c'])
ax.set_title('Fatigue effect on click rate\n(controlled: same suspicion threshold)')
ax.set_xlabel('Fatigue level')
ax.set_ylabel('Click rate')
ax.set_ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(ROOT / 'results' / 'fatigue_effect.png', dpi=150)
plt.show()

In [ ]:
# KSS + FPL trajectory (Three Process Model)
hours = np.arange(8.0, 17.0, 0.5)
archetypes = [
    ('Well-rested (sleep=8h, quality=5)', 5.0, 8.0, 7),
    ('Average (sleep=7.5h, quality=3)',   3.0, 7.5, 7),
    ('Sleep-deprived (sleep=5h, quality=1)', 1.0, 5.0, 7),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for label, sq, st, seed in archetypes:
    ag = Agent.random_agent(f'arch', seed=seed)
    ag.sleep_quality = sq
    ag.total_sleep_time = st
    kss_vals, fpl_vals = [], []
    for h in hours:
        ag.advance_workday(float(h))
        kss_vals.append(ag.compute_kss())
        fpl_vals.append(ag.compute_flawed_perception_level())
    axes[0].plot(hours, kss_vals, marker='.', label=label)
    axes[1].plot(hours, fpl_vals, marker='.', label=label)

axes[0].set_title('KSS — Karolinska Sleepiness Scale\n1=very alert, 9=very sleepy')
axes[0].set_ylim(1, 9); axes[0].set_ylabel('KSS')
axes[0].set_xticks([8,10,12,14,16]); axes[0].set_xticklabels(['8am','10am','12pm','2pm','4pm'])
axes[0].legend(fontsize=8)

axes[1].set_title('FPL — Flawed Perception Level\nHigher = more cues missed')
axes[1].set_ylim(0, 0.5); axes[1].set_ylabel('FPL')
axes[1].set_xticks([8,10,12,14,16]); axes[1].set_xticklabels(['8am','10am','12pm','2pm','4pm'])
axes[1].legend(fontsize=8)

plt.suptitle('Three Process Model: biological fatigue drives FPL (Åkerstedt)', fontsize=11)
plt.tight_layout()
plt.savefig(ROOT / 'results' / 'tpm_kss_fpl.png', dpi=150)
plt.show()

In [ ]:
# Agent trait correlation heatmap
agent_summary = (
    phishing_df.groupby('agent_id')
    .agg(
        click_rate=('clicked', 'mean'),
        avg_fatigue=('total_fatigue', 'mean'),
        avg_fpl=('fpl', 'mean'),
        age=('age', 'first'),
        education_level=('education_level', 'first'),
        suspicion_threshold=('suspicion_threshold', 'first'),
    ).reset_index()
)

corr_cols = ['click_rate', 'avg_fatigue', 'avg_fpl', 'age', 'education_level', 'suspicion_threshold']
corr = agent_summary[corr_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax, vmin=-1, vmax=1)
ax.set_title('Agent traits vs phishing click rate')
plt.tight_layout()
plt.savefig(ROOT / 'results' / 'trait_correlation.png', dpi=150)
plt.show()

In [ ]:
# Cue perception heatmap
import ast

all_cues = [
    'urgency', 'threats', 'generic_greeting', 'spelling_grammar',
    'emotional_appeal', 'too_good_true', 'personal_info',
    'suspicious_sender', 'suspicious_link'
]

def parse_cues(val):
    if isinstance(val, list): return val
    try: return ast.literal_eval(val)
    except: return []

phishing_df = phishing_df.copy()
phishing_df['cues_perceived'] = phishing_df['cues_perceived'].apply(parse_cues)
for cue in all_cues:
    phishing_df[f'perceived_{cue}'] = phishing_df['cues_perceived'].apply(lambda c: int(cue in c))

cue_by_source = (
    phishing_df.groupby('source')[[f'perceived_{c}' for c in all_cues]].mean() * 100
).round(1)
cue_by_source.columns = all_cues

fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(cue_by_source, annot=True, fmt='.0f', cmap='Blues', ax=ax)
ax.set_title('% of trials where cue was perceived — by email source')
plt.tight_layout()
plt.savefig(ROOT / 'results' / 'cue_heatmap.png', dpi=150)
plt.show()